# 05 · 测试与版本控制 pytest & Git

> **能力标签**：SH2（工具与可复现性）

## 目标 Objectives

完成本课后，你应该能够：

1. 解释为什么自动化测试对研究代码同样重要。
2. 理解 `pytest` 的**测试发现规则**：`test_*.py` / `*_test.py`，函数以 `test_` 开头。
3. 写含 `assert` 语句的简单测试，用 `pytest.raises` 测试异常。
4. 用 `@pytest.fixture` 提取测试公共前置数据；用 `@pytest.mark.parametrize` 参数化测试用例。
5. 描述 **RED → GREEN** TDD（测试驱动开发）工作流：先写失败的测试，再写代码让测试通过。
6. 说出 Git 基本命令（`add`、`commit`、`branch`、`switch`）的作用。
7. 解释为什么 Jupyter notebook 提交时必须**清除输出**（output-free 规则）。

> 对应能力：**SH2**


## 概念讲解 Concepts

### 为什么要写测试？

> "没有测试的代码是遗留代码。"——Michael Feathers

研究代码经常被认为"一次性用"而跳过测试，但：

- 实验结果要**可复现**，测试是可复现性的保证之一。
- 重构代码时，测试可以**立即发现回归（regression）**。
- 好的测试 = 最诚实的文档（展示函数的正确用法和边界情况）。

---

### `pytest` 基础

**测试发现规则**：

```
tests/
├── test_utils.py      ← 文件名以 test_ 开头
└── utils_test.py      ← 或以 _test 结尾
```

函数名以 `test_` 开头才会被收集：

```python
def test_add():
    assert 1 + 1 == 2         # 通过
    assert "ab" + "c" == "abc"  # 通过
```

运行方式：`python -m pytest` 或直接 `pytest`

---

### `assert` 语句

`pytest` 对 `assert` 做了特殊处理，失败时会打印详细差异：

```python
assert result == expected, "可选的失败信息"
assert len(items) > 0, f"items 为空：{items}"
```

---

### `pytest.raises`：测试异常

```python
import pytest

def divide(a, b):
    return a / b

def test_divide_by_zero():
    with pytest.raises(ZeroDivisionError):
        divide(10, 0)
```

---

### `@pytest.fixture`：公共前置数据

避免在每个测试里重复写同样的初始化代码：

```python
import pytest

@pytest.fixture
def sample_list():
    return [1, 2, 3, 4, 5]

def test_sum(sample_list):
    assert sum(sample_list) == 15

def test_len(sample_list):
    assert len(sample_list) == 5
```

`fixture` 的名字作为参数名传入测试函数，`pytest` 自动注入。

---

### `@pytest.mark.parametrize`：参数化测试

用一组输入-输出对批量生成测试用例：

```python
@pytest.mark.parametrize("a, b, expected", [
    (1, 2, 3),
    (0, 0, 0),
    (-1, 1, 0),
])
def test_add(a, b, expected):
    assert a + b == expected
```

这三行 `parametrize` 数据会生成三个独立测试用例。

---

### RED → GREEN TDD 工作流

1. **RED**：先写一个测试，运行——它应该**失败**（因为功能还没实现）。
2. **GREEN**：写最少的代码让测试通过——**不要过度设计**。
3. **REFACTOR**：在测试绿灯保护下重构代码，提高质量。
4. 回到步骤 1。

这个循环让你每次只聚焦一个小问题，错误更容易定位。

---

### Git 基本命令

| 命令 | 作用 |
|------|------|
| `git status` | 查看工作区状态（哪些文件改了） |
| `git add <file>` | 暂存文件（Stage） |
| `git commit -m "message"` | 提交暂存区内容，写提交信息 |
| `git log --oneline` | 查看简洁提交历史 |
| `git branch <name>` | 创建新分支 |
| `git switch <name>` | 切换到分支 |
| `git switch -c <name>` | 创建并切换到新分支 |
| `git diff` | 查看未暂存的改动 |

**原子提交**：每次提交只做一件事，提交信息说清楚"做了什么/为什么"。

---

### Notebook 输出清除规则（Output-Free Rule）

Jupyter notebook 的输出（图片、表格、打印结果）存储在 `.ipynb` 文件里，会导致：

- Git diff 噪音巨大（几十行 base64 图片数据）
- 合并冲突复杂
- 文件体积膨胀

**团队规范**：**提交前必须清除输出**：

```bash
jupyter nbconvert --clear-output --inplace notebook.ipynb
```

或在 Jupyter Lab：`Kernel → Restart Kernel and Clear All Outputs`，然后保存。


## 示例 Worked Example

下面在 notebook 里直接定义函数和测试，并用 `pytest` 的方式运行（借助 `ipytest` 思路，但这里不依赖外部包，直接调用函数验证）。


In [ ]:
# ── 定义被测试的函数 ────────────────────────────────────────────────────────

def add(a: float, b: float) -> float:
    """两数相加。"""
    return a + b


def divide(a: float, b: float) -> float:
    """两数相除；b 为 0 时抛 ValueError。"""
    if b == 0:
        raise ValueError("除数不能为 0")
    return a / b


print("函数定义完毕 ✓")


In [ ]:
# ── 模拟 pytest 断言行为 ────────────────────────────────────────────────────
# 真正运行时用：python -m pytest test_xxx.py
# 这里在 notebook 里直接验证，展示 pytest 测试的写法

import traceback


def run_test(fn):
    """简单测试运行器：调用测试函数，捕获并打印结果。"""
    try:
        fn()
        print(f"  PASS  {fn.__name__}")
    except AssertionError as e:
        print(f"  FAIL  {fn.__name__}: {e}")
    except Exception as e:
        print(f"  ERROR {fn.__name__}: {e}")


# ── 测试 add ────────────────────────────────────────────────────────────────
def test_add_basic():
    assert add(1, 2) == 3

def test_add_negative():
    assert add(-1, 1) == 0

def test_add_float():
    assert abs(add(0.1, 0.2) - 0.3) < 1e-9

# ── 测试 divide ─────────────────────────────────────────────────────────────
def test_divide_normal():
    assert divide(10, 2) == 5.0

def test_divide_by_zero():
    try:
        divide(10, 0)
        assert False, "应该抛出 ValueError"
    except ValueError:
        pass   # 期望行为

print("测试结果：")
for test_fn in [test_add_basic, test_add_negative, test_add_float,
                test_divide_normal, test_divide_by_zero]:
    run_test(test_fn)


In [ ]:
# ── 演示参数化测试的思路 ────────────────────────────────────────────────────
# 实际 pytest 里用 @pytest.mark.parametrize，这里手动迭代展示同样的效果

test_cases = [
    (1,  2,  3),
    (0,  0,  0),
    (-1, 1,  0),
    (100, -50, 50),
]

print("参数化测试 test_add[a, b, expected]：")
all_pass = True
for a, b, expected in test_cases:
    result = add(a, b)
    ok = result == expected
    status = "PASS" if ok else "FAIL"
    print(f"  {status}  add({a}, {b}) == {expected} → got {result}")
    all_pass = all_pass and ok

print()
print("全部通过：", all_pass)


## 动手 Exercise

下面有一个**故意写错的函数** `buggy_multiply`。

**任务**：

1. 先阅读函数代码，猜测 bug 在哪里。
2. 在"写测试"cell 里写一个测试，运行后它应该**失败**（RED）。
3. 在"修复"cell 里修复 `buggy_multiply`，再运行测试，让它**通过**（GREEN）。

这就是 TDD 的 RED → GREEN 循环。


In [ ]:
# ── 故意有 bug 的函数 ────────────────────────────────────────────────────────
def buggy_multiply(a: float, b: float) -> float:
    """两数相乘（有 bug）。"""
    return a + b   # Bug: 应该是乘法，写成了加法


In [ ]:
# ── 第一步：写测试（应该 FAIL）───────────────────────────────────────────────
def test_multiply():
    # TODO: 在这里写断言，验证 buggy_multiply(3, 4) == 12
    assert buggy_multiply(3, 4) == 12, f"期望 12，得到 {buggy_multiply(3, 4)}"

run_test(test_multiply)   # 应该打印 FAIL


In [ ]:
# ── 第二步：修复函数（让测试变 GREEN）────────────────────────────────────────
def buggy_multiply(a: float, b: float) -> float:
    """两数相乘（已修复）。"""
    # TODO: 改成正确的乘法
    return a + b

run_test(test_multiply)   # 修复后应该打印 PASS


## 延伸阅读 Further Reading

1. **pytest 官方文档**：<https://docs.pytest.org/>（中文版可 Google "pytest 中文文档"）
2. **Pro Git（开源书）**：<https://git-scm.com/book/zh/v2>——第 1-3 章涵盖日常工作流
3. **《测试驱动开发：案例》（TDD by Example）**——Kent Beck 著，了解 TDD 的哲学
4. **nbstripout**：自动清除 notebook 输出的 Git hook 工具：<https://github.com/kynan/nbstripout>
5. **pytest fixtures 深入**：<https://docs.pytest.org/en/stable/reference/fixtures.html>


## 关联练习 Related Assignments

本课关联所有 W1 练习包（通过 `check.py` 统一运行）：

```bash
# 对任意一个 W1 练习包运行自动评分器
python check.py w1-oop
python check.py w1-dataclass
python check.py w1-cli
python check.py w1-decorators
python check.py w1-generators
```

**Git 实践任务**（在你的 `work/w1/` 目录下）：

1. 创建 `feat/w1-oop` 分支，实现 `w1-oop` 练习，提交后合并回主分支。
2. 提交前用 `jupyter nbconvert --clear-output --inplace` 清除所有 notebook 输出。

> 能力标签：**SH2**
